In [ ]:
# ============================================================
# LOAN APPROVAL & RISK ANALYTICS
# STEP 5: MODEL TRAINING
# ============================================================

# ------------------------------------------------------------
# 1. Import Libraries
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 2. Create Models Directory
# ------------------------------------------------------------

os.makedirs("../models", exist_ok=True)


# ------------------------------------------------------------
# 3. Load Training and Testing Data
# ------------------------------------------------------------

train_data = pd.read_csv("../data/train_data.csv")
test_data = pd.read_csv("../data/test_data.csv")

print("Training Data Shape:", train_data.shape)
print("Testing Data Shape:", test_data.shape)


# ------------------------------------------------------------
# 4. Separate Features and Target
# ------------------------------------------------------------

X_train = train_data.drop("Loan_Status", axis=1)
y_train = train_data["Loan_Status"]

X_test = test_data.drop("Loan_Status", axis=1)
y_test = test_data["Loan_Status"]

print("\nX_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

print("\nTarget Distribution in Training Data:")
print(y_train.value_counts())


# ------------------------------------------------------------
# 5. Define Models
# ------------------------------------------------------------

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}


# ------------------------------------------------------------
# 6. Train Models
# ------------------------------------------------------------

results = []

trained_models = {}

for name, model in models.items():

    print("\n" + "=" * 60)
    print(f"Training: {name}")
    print("=" * 60)

    # Train
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    # Store results
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    # Store trained model
    trained_models[name] = model

    # Save model
    filename = name.lower().replace(" ", "_") + ".pkl"
    filepath = os.path.join("../models", filename)

    joblib.dump(model, filepath)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Model saved: {filepath}")


# ------------------------------------------------------------
# 7. Create Model Comparison DataFrame
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 8. Display Model Comparison
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)

display(
    results_df.style.format({
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 9. Save Model Comparison
# ------------------------------------------------------------

results_df.to_csv(
    "../data/model_comparison.csv",
    index=False
)

print("\nModel comparison saved to:")
print("../data/model_comparison.csv")


# ------------------------------------------------------------
# 10. Select Best Model
# ------------------------------------------------------------

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("\n" + "=" * 60)
print("BEST MODEL")
print("=" * 60)

print("Best Model:", best_model_name)
print(
    "Best F1 Score:",
    f"{results_df.iloc[0]['F1 Score']:.4f}"
)


# ------------------------------------------------------------
# 11. Save Best Model Separately
# ------------------------------------------------------------

joblib.dump(
    best_model,
    "../models/best_loan_model.pkl"
)

print("\nBest model saved as:")
print("../models/best_loan_model.pkl")


# ------------------------------------------------------------
# 12. Feature Importance
# ------------------------------------------------------------


# ------------------------------------------------------------
# 12. Feature Importance
# ------------------------------------------------------------

if hasattr(best_model, "feature_importances_"):

    feature_importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": best_model.feature_importances_
    })

    feature_importance["Coefficient"] = np.nan
    feature_importance["Direction"] = "Not applicable"

elif hasattr(best_model, "coef_"):

    coefficients = best_model.coef_[0]

    feature_importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Coefficient": coefficients,
        "Importance": np.abs(coefficients)
    })

    feature_importance["Direction"] = np.where(
        feature_importance["Coefficient"] > 0,
        "Positive",
        np.where(
            feature_importance["Coefficient"] < 0,
            "Negative",
            "Neutral"
        )
    )

else:
    feature_importance = pd.DataFrame()

if not feature_importance.empty:

    feature_importance = feature_importance.sort_values(
        by="Importance",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 60)
    print("TOP IMPORTANT FEATURES")
    print("=" * 60)

    display(feature_importance.head(15))

    feature_importance.to_csv(
        "../data/feature_importance.csv",
        index=False
    )

    print("\nFeature importance saved successfully!")

else:
    print("Feature importance is not supported by this model.")


# ------------------------------------------------------------
# 13. Final Validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
print("Number of features:", X_train.shape[1])
print("Models trained:", len(models))
print("Best model:", best_model_name)

print("\nFiles created:")
print("1. ../data/model_comparison.csv")
print("2. ../models/logistic_regression.pkl")
print("3. ../models/decision_tree.pkl")
print("4. ../models/random_forest.pkl")
print("5. ../models/xgboost.pkl")
print("6. ../models/best_loan_model.pkl")

print("\nModel training completed successfully! ✅")




Training Data Shape: (491, 21)
Testing Data Shape: (123, 21)

X_train Shape: (491, 20)
X_test Shape: (123, 20)

Target Distribution in Training Data:
Loan_Status
1    337
0    154
Name: count, dtype: int64

Training: Logistic Regression
Accuracy : 0.8699
Precision: 0.8485
Recall   : 0.9882
F1 Score : 0.9130
Model saved: ../models\logistic_regression.pkl

Training: Decision Tree
Accuracy : 0.8455
Precision: 0.8438
Recall   : 0.9529
F1 Score : 0.8950
Model saved: ../models\decision_tree.pkl

Training: Random Forest
Accuracy : 0.8618
Precision: 0.8469
Recall   : 0.9765
F1 Score : 0.9071
Model saved: ../models\random_forest.pkl

Training: XGBoost
Accuracy : 0.8293
Precision: 0.8556
Recall   : 0.9059
F1 Score : 0.8800
Model saved: ../models\xgboost.pkl


MODEL PERFORMANCE COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,86.99%,84.85%,98.82%,91.30%
1,Random Forest,86.18%,84.69%,97.65%,90.71%
2,Decision Tree,84.55%,84.38%,95.29%,89.50%
3,XGBoost,82.93%,85.56%,90.59%,88.00%



Model comparison saved to:
../data/model_comparison.csv

BEST MODEL
Best Model: Logistic Regression
Best F1 Score: 0.9130

Best model saved as:
../models/best_loan_model.pkl

FINAL VALIDATION
Training samples: 491
Testing samples : 123
Number of features: 20
Models trained: 4
Best model: Logistic Regression

Files created:
1. ../data/model_comparison.csv
2. ../models/logistic_regression.pkl
3. ../models/decision_tree.pkl
4. ../models/random_forest.pkl
5. ../models/xgboost.pkl
6. ../models/best_loan_model.pkl

Model training completed successfully! ✅


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

# Check that training variables are available
print("Best Model:", best_model_name)
print("Model Type:", type(best_model).__name__)

# Generate feature importance
if hasattr(best_model, "feature_importances_"):

    importance_values = best_model.feature_importances_

elif hasattr(best_model, "coef_"):

    importance_values = np.abs(best_model.coef_[0])

else:
    raise ValueError("Selected model does not support feature importance.")

# Create DataFrame
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance_values
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Save file
output_path = Path("../data/feature_importance.csv").resolve()

feature_importance.to_csv(output_path, index=False)

print("\nFeature Importance Generated Successfully!")
print("Saved Location:", output_path)
print("File Exists:", output_path.exists())

display(feature_importance.head(15))


NameError: name 'best_model_name' is not defined

In [3]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Load saved best model
model_path = Path("../models/best_loan_model.pkl")

best_model = joblib.load(model_path)

# Load training data
train_data = pd.read_csv("../data/train_data.csv")

X_train = train_data.drop("Loan_Status", axis=1)

print("Model Loaded:", type(best_model).__name__)
print("Number of Features:", X_train.shape[1])

# Calculate feature importance
if hasattr(best_model, "feature_importances_"):

    importance_values = best_model.feature_importances_

elif hasattr(best_model, "coef_"):

    importance_values = np.abs(best_model.coef_[0])

else:
    raise ValueError("Model does not support feature importance.")

# Create DataFrame
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance_values
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Save CSV
output_path = Path("../data/feature_importance.csv")

feature_importance.to_csv(output_path, index=False)

print("\nFeature Importance Generated!")
print("File Location:", output_path.resolve())
print("File Exists:", output_path.exists())

display(feature_importance.head(15))


Model Loaded: LogisticRegression
Number of Features: 20

Feature Importance Generated!
File Location: C:\Users\Princ\OneDrive\Desktop\Loan-Risk-Analytics\data\feature_importance.csv
File Exists: True


,Feature,Importance
0,Credit_History,1.267094
1,Property_Area_Semiurban,0.396567
2,LoanAmountLog,0.291876
3,LoanIncomeRatio,0.271511
4,Married_Yes,0.259180
5,IncomePerDependent,0.198070
6,Dependents_1,0.178294
7,CoapplicantIncome,0.161064
8,Education_Not Graduate,0.155246
9,TotalIncomeLog,0.147036
